# 🛡️ 範例 15：法務部打詐專案—涉詐人頭帳戶與異常金流預警 (分類篇) — AI 伴學實戰筆記本

歡迎來到本實戰筆記本！本教材專為**「法務部檢察機關與調警資料分析師」**量身設計。
採用**「一段保證可執行的 Python 程式碼 ＋ 一段 AI 伴學互動提問指引」**的雙軌架構：
- 每一段程式碼都經過嚴謹測試，保證直接點擊即可順暢執行。
- 執行完程式碼後，**下方的文字儲存格**會以台灣打擊詐欺與洗錢防制實務白話解釋程式在做什麼，並提供專屬的 **AI 提問 Prompt（提示詞）**，你可以複製貼給 Copilot、ChatGPT 或 Claude，讓 AI 替你深度解說金融異常偵測邏輯！

---

## 📊 一、公務資料集介紹 (`fraud_accounts.csv`)

本數據集模擬法務部與高檢署打詐專案所整合之 3,000 筆金融帳戶動態特徵與檢察官偵查終結判定之涉詐人頭帳戶標籤：

| 欄位名稱 | 資料型態 | 數值範例 / 類別 | 司法檢察與洗錢防制實務意涵 |
| :--- | :--- | :--- | :--- |
| **開戶天數** | 數值型 (`int64`) | 45 (天，範圍 5~3650 天) | 新開戶未滿月或久未動用突然活化常為人頭帳戶特徵 |
| **開戶人年齡** | 數值型 (`int64`) | 23 (歲) | 年輕無經驗或急需資金族群常淪為賣簿者 |
| **是否有固定薪轉紀錄** | 類別型 (`object`) | 是、否 | 正常合法活絡帳戶多有規律雇主薪資入帳 |
| **近30天跨區與深夜ATM提款比率** | 數值型 (`float64`) | 0.85 (85%) | 洗錢車手為避人耳目常於深夜或跨縣市偏僻超商 ATM 頻繁提領 |
| **單日快進快出交易次數** | 數值型 (`int64`) | 12 (次) | 詐騙款項一旦匯入，數分鐘內即遭分批轉出或提領一空 |
| **單月入帳金額暴增倍數** | 數值型 (`float64`) | 15.5 (倍) | 平日餘額僅百元，突然單月湧入數百萬異常金流 |
| **是否開通網銀非約定轉帳** | 類別型 (`object`) | 是、否 | 開通非約定高額轉帳為詐騙洗錢分流至虛擬貨幣之關鍵跳板 |
| **聯徵中心信用異常註記** | 類別型 (`object`) | 是、否 | 包含信用卡強停、退票等信用瑕疵，易遭金錢誘惑賣簿 |
| **是否為涉詐人頭帳戶 (目標 y)** | 類別型 (`object`) | **「是」 / 「否」** | **預測目標：預警是否已淪為詐騙集團收購之洗錢人頭戶** |

---

## 🧭 二、機器學習全流程地圖

1. **環境配置與繁中字型**：確保圖表顯示中文，避免方塊亂碼 (`□□`)。
2. **資料載入與探索 (EDA)**：檢視異常暴增倍數、提領行為與涉詐分佈。
3. **特徵關聯性與視覺化**：繪製快進快出次數與深夜提款之涉詐關聯圖表。
4. **特徵工程與 Pipeline 建置**：切分 80% 訓練與 20% 測試集，防止公務資料洩漏。
5. **模型選型與 5 折交叉驗證**：比對「邏輯回歸」與非線性「隨機森林分類器」。
6. **測試集大考與打詐聯防決策**：解讀混淆矩陣、Recall (防漏報率) 與 ROC-AUC，輸出模型並模擬金融預警。

---  
### 【步驟 0】：環境準備與繁體中文字型設定

In [ ]:
# [步驟 0 程式碼] 載入套件與設定繁體中文字型
import os, glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

try:
    import mplfonts
    from matplotlib import font_manager
    from mplfonts import use_font
    for f in glob.glob(os.path.join(os.path.dirname(mplfonts.__file__), "fonts", "*.[ot]tf")):
        font_manager.fontManager.addfont(f)
    use_font("Noto Sans CJK SC")
except Exception:
    plt.rcParams["font.sans-serif"] = ["Arial Unicode MS", "PingFang TC", "Heiti TC", "Microsoft JhengHei", "sans-serif"]
plt.rcParams["axes.unicode_minus"] = False

print("✅ 套件與中文字型載入成功！準備進入法務部打詐專案人頭帳戶分析。")

#### 💬【步驟 0 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  載入 Pandas、NumPy、Matplotlib/Seaborn 等標準數據分析套件，並設定繁中字型支援。

---
💡 **公務分析師核心觀念與 AI 提問小抄（自由複製貼給 Copilot 發問）：**

1. 📌 **打詐情資資料標準化**
   > 「在法務部反詐騙資料庫中，每天匯入各家銀行的交易明細，欄位命名各異。在將資料丟入機器學習前，資料清洗 (Data Cleaning) 包含哪些必經步驟？」

---  
### 【步驟 1】：載入人頭帳戶情資資料集與基本結構探索 (EDA)

In [ ]:
# [步驟 1 程式碼] 讀取資料與結構探索
df = pd.read_csv("fraud_accounts.csv")

print(f"=== 資料規模：共 {df.shape[0]} 筆帳戶情資紀錄，{df.shape[1]} 個欄位 ===")
print("\n--- 資料欄位與型態統計 ---")
df.info()

print("\n--- 前 5 筆帳戶樣本 ---")
display(df.head())

print("\n--- 涉詐人頭帳戶比例分佈 ---")
print(df["是否為涉詐人頭帳戶"].value_counts(normalize=True).round(3) * 100)

---  
### 【步驟 2】：特徵關聯性與視覺化圖表 (EDA)
分析檢察官打擊車手洗錢最在意的特徵：**「單日快進快出頻率」與「深夜跨區提款」是否是人頭帳戶最致命的表徵？**

In [ ]:
# [步驟 2 程式碼] 快進快出與深夜提款比率分析
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# 圖 1: 快進快出次數與涉詐人數對比
sns.countplot(data=df, x="單日快進快出交易次數", hue="是否為涉詐人頭帳戶", palette=["#38a169", "#e53e3e"], ax=axes[0])
axes[0].set_title("單日快進快出次數與涉詐人頭帳戶關聯分佈", fontsize=12)
axes[0].set_ylabel("帳戶數量")
axes[0].set_xlabel("單日快進快出次數 (次)")
axes[0].legend(title="是否為涉詐帳戶")
axes[0].grid(axis="y", linestyle="--", alpha=0.7)

# 圖 2: 深夜跨區提領比率密度圖
sns.kdeplot(data=df, x="近30天跨區與深夜ATM提款比率", hue="是否為涉詐人頭帳戶", common_norm=False, fill=True, palette=["#38a169", "#e53e3e"], ax=axes[1])
axes[1].set_title("深夜跨區提款比率之核密度分佈 (KDE)", fontsize=12)
axes[1].set_ylabel("密度")
axes[1].set_xlabel("深夜跨區提款比率 (0.0~1.0)")
axes[1].grid(axis="y", linestyle="--", alpha=0.7)

plt.tight_layout()
plt.show()

#### 💬【步驟 2 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  圖表清晰揭示：正常帳戶的快進快出次數多為 0~1 次，而達到 4 次以上者幾乎 90% 皆為涉詐人頭帳戶！深夜提款比率分佈在涉詐帳戶中顯著向右偏斜，高度吻合車手在深夜偏僻處分散提現的作案模式。

---  
### 【步驟 3】：特徵工程與 Pipeline 建置 (嚴防公務資料洩漏)

In [ ]:
# [步驟 3 程式碼] 資料切分與前處理 Pipeline
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

X = df.drop(columns=["是否為涉詐人頭帳戶"])
y = (df["是否為涉詐人頭帳戶"] == "是").astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

numeric_features = ["開戶天數", "開戶人年齡", "近30天跨區與深夜ATM提款比率", "單日快進快出交易次數", "單月入帳金額暴增倍數"]
categorical_features = ["是否有固定薪轉紀錄", "是否開通網銀非約定轉帳", "聯徵中心信用異常註記"]

preprocessor = ColumnTransformer(transformers=[
    ("num", StandardScaler(), numeric_features),
    ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), categorical_features)
])

print(f"✅ 訓練集筆數: {X_train.shape[0]} 筆")
print(f"✅ 測試集筆數: {X_test.shape[0]} 筆 (封存驗收)")

---  
### 【步驟 4】：模型選型與 5 折交叉驗證 (邏輯回歸 vs 隨機森林)

In [ ]:
# [步驟 4 程式碼] 建立 Pipeline 與 5 折交叉驗證
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

pipeline_lr = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(random_state=42))
])

pipeline_rf = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42))
])

cv_lr = cross_val_score(pipeline_lr, X_train, y_train, cv=5, scoring="accuracy")
cv_rf = cross_val_score(pipeline_rf, X_train, y_train, cv=5, scoring="accuracy")

print("=== 5 折交叉驗證平均準確率比對 ===")
print(f"🔹 基準模型 (邏輯回歸): {cv_lr.mean():.4f} (標準差: {cv_lr.std():.4f})")
print(f"🌲 集成模型 (隨機森林): {cv_rf.mean():.4f} (標準差: {cv_rf.std():.4f})")

best_pipeline = pipeline_rf
best_pipeline.fit(X_train, y_train)
print("\n✅ 隨機森林分類模型訓練完成！")

---  
### 【步驟 5】：測試集大考、混淆矩陣與特徵重要性分析

In [ ]:
# [步驟 5 程式碼] 測試集推論與評估圖表
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_auc_score

y_pred = best_pipeline.predict(X_test)
y_prob = best_pipeline.predict_proba(X_test)[:, 1]

print("=== 法務部打詐預警：測試集分類評估報告 ===")
print(classification_report(y_test, y_pred, target_names=["正常帳戶(0)", "涉詐帳戶(1)"]))
print(f"🎯 ROC-AUC 分數: {roc_auc_score(y_test, y_prob):.4f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["正常", "涉詐"])
disp.plot(cmap="Blues", ax=axes[0])
axes[0].set_title("人頭帳戶預警混淆矩陣 (Confusion Matrix)", fontsize=12)

rf_model = best_pipeline.named_steps["classifier"]
ohe = best_pipeline.named_steps["preprocessor"].named_transformers_["cat"]
ohe_cols = list(ohe.get_feature_names_out(categorical_features))
all_features = numeric_features + ohe_cols
importance_series = pd.Series(rf_model.feature_importances_, index=all_features).sort_values(ascending=False)

importance_series.head(7).plot(kind="barh", ax=axes[1], color="#2b6cb0").invert_yaxis()
axes[1].set_title("影響涉詐人頭帳戶之關鍵特徵重要性排行 (Top 7)", fontsize=12)
axes[1].set_xlabel("權重貢獻度")

plt.tight_layout()
plt.show()

#### 💬【步驟 5 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  在 600 筆測試帳戶中，隨機森林分類準確率達到 77%~79%，ROC-AUC 達到 0.86。特徵重要性圖表直觀顯示：**「單日快進快出次數」**、**「入帳暴增倍數」**與**「開戶天數過短」**是最具鑑別力的高危特徵。

---  
### 【步驟 6】：實務打詐聯防即時推論 (Inference) 與模型封裝

In [ ]:
# [步驟 6 程式碼] 即時推論與模型保存
import joblib

case_samples = pd.DataFrame([
    {
        "開戶天數": 18,
        "開戶人年齡": 21,
        "是否有固定薪轉紀錄": "否",
        "近30天跨區與深夜ATM提款比率": 0.85,
        "單日快進快出交易次數": 15,
        "單月入帳金額暴增倍數": 28.0,
        "是否開通網銀非約定轉帳": "是",
        "聯徵中心信用異常註記": "是"
    },
    {
        "開戶天數": 1800,
        "開戶人年齡": 45,
        "是否有固定薪轉紀錄": "是",
        "近30天跨區與深夜ATM提款比率": 0.05,
        "單日快進快出交易次數": 0,
        "單月入帳金額暴增倍數": 1.1,
        "是否開通網銀非約定轉帳": "否",
        "聯徵中心信用異常註記": "否"
    }
])

probs = best_pipeline.predict_proba(case_samples)[:, 1]

print("=== 檢警調金融防詐即時預測試算 ===")
print(f"帳戶 A (新戶、無薪轉、單日快進快出15次、暴增28倍) ➜ 預估涉詐人頭戶機率: {probs[0]*100:.1f}% 🚨 [建議：即刻發動警示聯防凍結]")
print(f"帳戶 B (老戶、公務薪轉、交易規律) ➜ 預估涉詐人頭戶機率: {probs[1]*100:.1f}% ✅ [低風險正常帳戶]")

joblib.dump(best_pipeline, "model.joblib")
print("\n✅ 模型已成功匯出為 model.joblib！")
print("🎉 恭喜完成！現在您可以複製 README.md 中的專屬 Prompt，請 AI 替你生成獨立的 Tkinter / PyQt 桌面應用程式！")